In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
        text = f.read()

In [3]:
len(text)

1115394

In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chars = sorted(list(set(text)))
vocab_size  = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
# 分词
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for i,s in enumerate(chars)}

encoder = lambda s : [stoi[i] for i in s]
decoder = lambda l : ''.join([itos[i] for i in l]) 

print(encoder("hi i'm here"))
print(decoder(encoder("hi i'm here")))

[46, 47, 1, 47, 5, 51, 1, 46, 43, 56, 43]
hi i'm here


In [7]:
data = torch.tensor(encoder(text), dtype=torch.long)
print(data.shape)


torch.Size([1115394])


In [8]:
n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]
block_size = 8

In [9]:
ix = torch.randint(44,(2,))
torch.stack([ix,ix])
block_size = 8
batch_size = 4

def get_batch(input):
    data = train_data if input == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

x, y = get_batch("train")

print("x is :")
print(x)

print("y is :")
print(y)

print('----------------------------')
# for i in range(batch_size):
    # for j in range(block_size):
        # print(f'input {x[i, :j+1]} we get: {y[i,j]}')
print(x.shape, y.shape)

x is :
tensor([[ 6,  1, 52, 39, 49, 43, 42,  6],
        [ 1, 63, 53, 59,  1, 58, 47, 50],
        [13, 45, 39, 47, 52, 57, 58,  1],
        [41, 46,  1, 58, 46, 43, 63,  1]])
y is :
tensor([[ 1, 52, 39, 49, 43, 42,  6,  1],
        [63, 53, 59,  1, 58, 47, 50, 50],
        [45, 39, 47, 52, 57, 58,  1, 51],
        [46,  1, 58, 46, 43, 63,  1, 42]])
----------------------------
torch.Size([4, 8]) torch.Size([4, 8])


In [10]:
import torch
torch.randn(3,5)
torch.randint(5, (3,))

tensor([3, 4, 3])

In [23]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

**以上是分词和分割数据集，接下来是模型与训练推理**

In [20]:
import torch.nn as nn

#定义常量
n_embed = 32
head_size = 32
n_block = 6

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.key   = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):# x : (B,T,n_embed)
        B, T, C = x.shape
        
        q = self.query(x) # (B,T,head_size)
        k = self.key(x)   # (B,T,head_size)
        v = self.value(x) # (B,T,head_size)

        score = q @ k.transpose(-2,-1)
        wei = score / ( k.shape[-1]**0.5) # (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = torch.softmax(wei,dim = -1)#(B, T, T)
        out = wei @ v #(B,T,head_size)
        return out

class Muti_Head_Attention(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.mha = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.proj = nn.Linear(n_embed, n_embed)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.mha], dim=-1)
        out = self.proj(x)
        return out
    
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.ffwd = nn.Sequential(
            nn.Linear(n_embed, 4*n_embed),
            nn.ReLU(),
            nn.Linear(4*n_embed, n_embed),
        )

    def forward(self, x):
        out = self.ffwd(x)
        return out
    
class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()
        self.mha = Muti_Head_Attention(n_embed, n_head)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ffwd = FeedForward(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
    
    def forward(self, x):
        x = x + self.mha(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
        

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.position_embedding = nn.Embedding(block_size, n_embed)
        self.block = nn.Sequential(*[Block(n_embed, n_head=4) for _ in range(n_block)])
        self.lf = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(head_size, vocab_size)

    def forward(self, idx, target=None):    
        B,T = idx.shape
        token_emb = self.token_embedding(idx)
        position_emb = self.position_embedding(torch.arange(T))
        x = token_emb + position_emb
        x = self.block(x)
        x = self.lf(x)
        logits = self.lm_head(x)

        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)
        
        return logits, loss
    
    def generate(self, idx, max_block = 100):
        for _ in range(max_block):
            id_cond = idx[: , -block_size:]
            logits, loss = self(id_cond)
            logit = logits[:, -1, :]
            probs =F.softmax(logit, dim=-1)
            id_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, id_next),dim=1)
        return idx

model = BigramLanguageModel(vocab_size)
x, y = get_batch('train')
logits, loss = model(x, y)
print(loss)


tensor(4.4432, grad_fn=<NllLossBackward0>)


In [25]:
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
eval_iters = 200
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


step 0: train loss 2.5286, val loss 2.4933
step 500: train loss 2.5258, val loss 2.5504
step 1000: train loss 2.5412, val loss 2.5349
step 1500: train loss 2.5391, val loss 2.5489
step 2000: train loss 2.5478, val loss 2.5357
step 2500: train loss 2.5339, val loss 2.5374
step 3000: train loss 2.5110, val loss 2.5173
step 3500: train loss 2.5164, val loss 2.5562
step 4000: train loss 2.5268, val loss 2.5199
step 4500: train loss 2.5022, val loss 2.5093
step 4999: train loss 2.4968, val loss 2.5293


In [22]:
context = torch.zeros((1, 1), dtype=torch.long)
print(decoder(model.generate(context, max_block=100)[0].tolist()))


pousherlll Sou endounistcesa, thar,
An be, bongrativeshin'sthe, hioo seag havesed turd'lor was is, n
